# 15.8 為什麼 token 都跑去同一位 expert？


準備三位expert，並不保證它們都得到工作。router如果一開始稍微偏向第一位，就可能讓它接到更多token、得到更多學習機會；其他兩位很少更新，表現差距又讓router繼續偏向第一位。這種只使用少數甚至一位expert的狀態叫routing collapse（路由塌縮）。要診斷它，不能只看模型宣告有幾位expert，還要數實際選擇。

為了看清[比例與排名](https://birdhackor.github.io/tiny-perceptron-vlm/15.4.html)的差別，固定使用top-1，讓十二個token都有相同分數 `[8,0,0]`。我們預期十二筆都去第一位，再把優勢縮到0.1，看排名是否真的分散。這是刻意製造的偏斜，不靠某次隨機訓練碰巧發生。softmax後第一位的比例約0.9993，其餘各0.0003；十二筆選擇都會落在索引0，實際工作數為 `[12,0,0]`。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
import torch

for advantage in (8.0, 0.1):
    scores = torch.tensor([[advantage, 0.0, 0.0]]).repeat(12, 1)
    prob = scores.softmax(-1)
    chosen = prob.topk(1, dim=-1).indices
    counts = torch.bincount(chosen.flatten(), minlength=3)
    print("分數優勢", advantage)
    print("平均比例", prob.mean(0).round(decimals=4).tolist())
    print("工作次數", counts.tolist())

分數優勢 8.0
平均比例 [0.9993000030517578, 0.0003000000142492354, 0.0003000000142492354]
工作次數 [12, 0, 0]
分數優勢 0.1
平均比例 [0.35589998960494995, 0.32199999690055847, 0.32199999690055847]
工作次數 [12, 0, 0]


`repeat(12,1)`把一列分數重複十二次。`flatten`將十二列一欄的索引攤成清單，`bincount`數每個非負整數出現幾次；`minlength=3`保留未被選中的expert欄位，不讓它們從報告消失。優勢8時平均比例約 `[0.9993,0.0003,0.0003]`，工作次數為 `[12,0,0]`。

第二輪把優勢降到0.1，平均比例變成約 `[0.3559,0.3220,0.3220]`，看起來已接近均勻；工作次數卻仍是 `[12,0,0]`。原因是top-1只看排名，第一位每次還是最大。這個對照非常重要：平均router機率與實際負載是不同統計。負載可用各位被選次數除以總選擇次數，這裡依然是 `[1,0,0]`，不是上面的三項機率。

均勻負載也不等於品質最好；某些輸入確實可能更適合某位expert。但如果所有種類的token長期集中在同一位，其他參數沒有發揮作用，系統還可能因單位工作過多而塞住。因此應同時記錄不同時間、不同資料類型的選擇分布和任務誤差。這個受控反例只證明偏斜如何形成與如何辨識，沒有宣稱真實訓練一定會塌縮。

練習只把第二個expert的分數也設成0.2，也就是每列 `[0.1,0.2,0]`。先預測十二筆改送索引1，而非自動平均分給三位，再執行核對。接著用兩組不同分數各重複六列，你應能用bincount看見 `[6,6,0]`。分散工作需要不同輸入產生不同排名，不能只讓每一列的比例更平坦。

<details>
<summary>選讀：實際分派次數與負載熵的算法</summary>

下面的實測會提到「輔助項」：它是訓練時額外加入、用來鼓勵router較均衡分派工作的誤差。「輔助係數」決定把這個額外項乘上多少，再加到文字預測誤差上；係數0表示這次沒有把它加入訓練目標。完整公式留到下一節[15.9](https://birdhackor.github.io/tiny-perceptron-vlm/15.9.html)。

實測top-1、輔助係數0的第二層，四位在全部有效驗證輸入上收到 `[8536,25630,4875,215]` 筆，占39,256筆的 `[21.74%,65.29%,12.42%,0.55%]`，完整比較見[15.13](https://birdhackor.github.io/tiny-perceptron-vlm/15.13.html)。這是偏斜使用，卻還不是全部工作都去一位。可再用「負載熵」把份額的分散程度概括成一個數字：每位的份額f各算 `-f*log(f)`，再把各項相加。這裡log是自然對數，問「約2.718的數e取幾次方會得到f」，Python用 `math.log(f)`計算。份額為0的項直接記0，不呼叫 `log(0)`；份額越接近0，這一項的貢獻也越接近0。

例如四位均勻時各占0.25，`log(0.25)`約-1.3863，各項為 `-0.25×(-1.3863)=0.3466`，四項合計約1.3863，也就是 `log(4)`。用熵除以這個均勻值，便把均勻負載記成1。若份額為 `[1,0,0,0]`，第一項因 `log(1)=0`而為0，其餘按零份額規則也為0，合計與正規化結果都是0。這兩端分別表示全去一位與整體份額均勻。

下面用實際次數重算，避免只拿四捨五入的百分比代入。`fractions`把各次數除以總次數，得到份額；`terms`逐項計算上述貢獻，`if f > 0 else 0.0`處理零份額；最後相加並除以四位均勻時的熵，印出0.64755。


In [3]:
import math

counts = [8536, 25630, 4875, 215]
fractions = [count / sum(counts) for count in counts]
terms = [-f * math.log(f) if f > 0 else 0.0 for f in fractions]
print(round(sum(terms) / math.log(len(counts)), 5))

0.64755


這個熵彙整整份驗證集，不保證每一批、每一類文字都均勻；不同批各自集中到不同expert，也可能加總成高熵。報告因此保留各位次數、平均機率與逐批輔助項的加權平均值。這個平均先把每批的輔助項乘上該批有效輸入數，全部相加後再除以有效輸入總數；有效輸入較多的批次占比較大。閱讀結果時應一起看這些統計。加上輔助目標後會怎樣，接著看[15.9](https://birdhackor.github.io/tiny-perceptron-vlm/15.9.html)。

</details>
